In [7]:
from abc import ABC, abstractmethod
class Media(ABC):
    def __init__(self, title, genre, year, rating):
        self._title = title    
        self._genre = genre
        self._year = year
        self._rating = rating

    def get_title(self): 
        return self._title
    def get_genre(self): 
        return self._genre
    def get_year(self): 
        return self._year
    def get_rating(self): 
        return self._rating
    def get_info(self): 
        return f"{self._title} | {self._genre} | {self._year} | {self._rating}"
    @abstractmethod
    def display(self): 
        pass

In [8]:
class Movie(Media):
    def __init__(self, title, genre, year, rating, duration):
        super().__init__(title, genre, year, rating)
        self._duration = duration
    def get_duration(self): 
        return self._duration
    def display(self): 
        return f"Movie: {self.get_info()} | Duration: {self._duration} Hours"

In [9]:
class Series(Media):
    def __init__(self, title, genre, year, rating, seasons):
        super().__init__(title, genre, year, rating)
        self._seasons = seasons
    def get_seasons(self): 
        return self._seasons
    def display(self): 
        return f"Series: {self.get_info()} | Seasons: {self._seasons}"

In [10]:
class MovieManager:
    def __init__(self):
        self.__media_list = []  
    def add(self, media):
        if isinstance(media, Media): 
            self.__media_list.append(media)
        else: 
            raise TypeError("Only Media objects allowed")
    def get_all(self): 
        return self.__media_list
    def search(self, keyword): 
        return [m for m in self.__media_list if keyword.lower() in m.get_title().lower()]
    def delete(self, media):
        if media in self.__media_list: 
            self.__media_list.remove(media)
    def update(self, index, new_media):
        if 0 <= index < len(self.__media_list): 
            self.__media_list[index] = new_media

In [11]:
import tkinter as tk
from tkinter import ttk, messagebox


class MediaGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Media Management System")
        self.root.geometry("900x600")
        self.bg_color = "#1e1e1e"
        self.fg_color = "#ffffff"
        self.card_color = "#2d2d2d"
        self.accent_color = "#3d3d3d"
        self.header_color = "#111111"
        self.root.configure(bg=self.bg_color)
        self.manager = MovieManager()
        self.selected_index = -1
        self.type_var = tk.StringVar(value="Movie")
        self.title_var = tk.StringVar()
        self.genre_var = tk.StringVar()
        self.year_var = tk.StringVar()
        self.rating_var = tk.StringVar()
        self.extra_var = tk.StringVar()
        self.search_var = tk.StringVar()
        self.setup_styles()
        self.setup_ui()

    def setup_styles(self):
        style = ttk.Style()
        style.theme_use("clam")
        style.configure(
            "Treeview",
            background=self.card_color,
            foreground=self.fg_color,
            fieldbackground=self.card_color,
            borderwidth=0
        )

        style.map("Treeview", background=[('selected', '#4a4a4a')])
        style.configure(
            "Treeview.Heading",
            background=self.header_color,
            foreground=self.fg_color,
            relief="flat"
        )

        style.configure(
            "TCombobox",
            fieldbackground=self.accent_color,
            background=self.accent_color,
            foreground=self.fg_color
        )

    def setup_ui(self):
        header = tk.Label(
            self.root,
            text="Media Library Management",
            font=("Arial", 22, "bold"),
            bg=self.header_color,
            fg=self.fg_color,
            pady=15
        )
        header.pack(fill=tk.X)
        main_container = tk.Frame(
            self.root,
            bg=self.bg_color,
            padx=20,
            pady=20
        )
        main_container.pack(fill=tk.BOTH, expand=True)
        input_frame = tk.LabelFrame(
            main_container,
            text=" Media Details ",
            bg=self.bg_color,
            fg=self.fg_color,
            font=("Arial", 11, "bold"),
            padx=15,
            pady=15
        )
        input_frame.place(x=0, y=0, width=350, height=480)
        fields = [
            ("Type:", self.type_var),
            ("Title:", self.title_var),
            ("Genre:", self.genre_var),
            ("Year:", self.year_var),
            ("Rating:", self.rating_var)
        ]

        for i, (label, var) in enumerate(fields):
            tk.Label(
                input_frame,
                text=label,
                bg=self.bg_color,
                fg=self.fg_color,
                font=("Arial", 10)
            ).grid(row=i, column=0, sticky="w", pady=8)

            if label == "Type:":
                combo = ttk.Combobox(
                    input_frame,
                    textvariable=var,
                    values=["Movie", "Series"],
                    state="readonly",
                    width=23
                )
                combo.grid(row=i, column=1, pady=8)
                combo.bind("<<ComboboxSelected>>", self.update_labels)

            else:
                tk.Entry(
                    input_frame,
                    textvariable=var,
                    width=25,
                    bg=self.accent_color,
                    fg=self.fg_color,
                    insertbackground="white",
                    relief="flat"
                ).grid(row=i, column=1, pady=8)

        self.lbl_extra = tk.Label(
            input_frame,
            text="Duration (Hours):",
            bg=self.bg_color,
            fg=self.fg_color,
            font=("Arial", 10)
        )
        self.lbl_extra.grid(row=5, column=0, sticky="w", pady=8)

        tk.Entry(
            input_frame,
            textvariable=self.extra_var,
            width=25,
            bg=self.accent_color,
            fg=self.fg_color,
            insertbackground="white",
            relief="flat"
        ).grid(row=5, column=1, pady=8)
        btn_frame = tk.Frame(input_frame, bg=self.bg_color)
        btn_frame.place(x=5, y=320, width=310)

        btn_config = {
            "font": ("Arial", 10, "bold"),
            "fg": "white",
            "relief": "flat",
            "width": 10,
            "pady": 5
        }

        tk.Button(btn_frame, text="Add", bg="#2e7d32",
                  command=self.add_item, **btn_config).grid(row=0, column=0, padx=5, pady=5)

        tk.Button(btn_frame, text="Update", bg="#1565c0",
                  command=self.update_item, **btn_config).grid(row=0, column=1, padx=5, pady=5)

        tk.Button(btn_frame, text="Delete", bg="#c62828",
                  command=self.delete_item, **btn_config).grid(row=1, column=0, padx=5, pady=5)

        tk.Button(btn_frame, text="Clear", bg="#616161",
                  command=self.clear_fields, **btn_config).grid(row=1, column=1, padx=5, pady=5)

        display_frame = tk.Frame(main_container, bg=self.bg_color)
        display_frame.place(x=370, y=0, width=490, height=480)
        search_f = tk.Frame(display_frame, bg=self.bg_color)
        search_f.pack(fill=tk.X, pady=(0, 10))

        tk.Entry(
            search_f,
            textvariable=self.search_var,
            width=25,
            bg=self.accent_color,
            fg=self.fg_color,
            insertbackground="white",
            relief="flat"
        ).pack(side=tk.LEFT, padx=5)

        tk.Button(search_f, text="Search", bg="#424242",
                  fg="white", relief="flat",
                  padx=10, command=self.search_items).pack(side=tk.LEFT, padx=2)

        tk.Button(search_f, text="All", bg="#424242",
                  fg="white", relief="flat",
                  padx=10, command=self.refresh_table).pack(side=tk.LEFT, padx=2)

        self.table = ttk.Treeview(
            display_frame,
            columns=("type", "title", "genre", "year", "rating", "extra"),
            show="headings"
        )

        headings = [
            ("type", "Type"),
            ("title", "Title"),
            ("genre", "Genre"),
            ("year", "Year"),
            ("rating", "Rating"),
            ("extra", "Info")
        ]

        for col_id, txt in headings:
            self.table.heading(col_id, text=txt)
            self.table.column(col_id, width=75, anchor="center")

        self.table.pack(fill=tk.BOTH, expand=True)
        self.table.bind("<<TreeviewSelect>>", self.on_select)

    def update_labels(self, e=None):
        if self.type_var.get() == "Movie":
            self.lbl_extra.config(text="Duration (Hours):")
        else:
            self.lbl_extra.config(text="Seasons:")

    def clear_fields(self):
        self.title_var.set("")
        self.genre_var.set("")
        self.year_var.set("")
        self.rating_var.set("")
        self.extra_var.set("")
        self.selected_index = -1

    def add_item(self):
        t = self.title_var.get()
        g = self.genre_var.get()
        y = self.year_var.get()
        r = self.rating_var.get()
        e = self.extra_var.get()

        if not all([t, g, y, r, e]):
            return messagebox.showwarning("Warning", "Fill all fields")

        if self.type_var.get() == "Movie":
            item = Movie(t, g, y, r, e)
        else:
            item = Series(t, g, y, r, e)

        self.manager.add(item)
        self.refresh_table()
        self.clear_fields()

    def refresh_table(self, data=None):
        self.table.delete(*self.table.get_children())
        items = data if data is not None else self.manager.get_all()

        for item in items:
            m_type = "Movie" if isinstance(item, Movie) else "Series"
            if m_type == "Movie":
                extra = item.get_duration()
            else:
                extra = item.get_seasons()

            self.table.insert(
                "",
                tk.END,
                values=(
                    m_type,
                    item.get_title(),
                    item.get_genre(),
                    item.get_year(),
                    item.get_rating(),
                    extra
                )
            )

    def on_select(self, e):
        sel = self.table.selection()
        if not sel:
            return

        vals = self.table.item(sel)['values']
        self.selected_index = self.table.index(sel)
        self.type_var.set(vals[0])
        self.title_var.set(vals[1])
        self.genre_var.set(vals[2])
        self.year_var.set(vals[3])
        self.rating_var.set(vals[4])
        self.extra_var.set(vals[5])
        self.update_labels()

    def update_item(self):
        if self.selected_index == -1:
            return

        t = self.title_var.get()
        g = self.genre_var.get()
        y = self.year_var.get()
        r = self.rating_var.get()
        e = self.extra_var.get()

        if self.type_var.get() == "Movie":
            new_item = Movie(t, g, y, r, e)
        else:
            new_item = Series(t, g, y, r, e)

        self.manager.update(self.selected_index, new_item)
        self.refresh_table()
        self.clear_fields()

    def delete_item(self):
        if self.selected_index == -1:
            return

        if messagebox.askyesno("Confirm", "Delete?"):
            item = self.manager.get_all()[self.selected_index]
            self.manager.delete(item)

            self.refresh_table()
            self.clear_fields()

    def search_items(self):
        query = self.search_var.get()
        results = self.manager.search(query)
        self.refresh_table(results)

if __name__ == "__main__":
    root = tk.Tk()
    gui = MediaGUI(root)
    root.mainloop()